# SEBRT-based-Model

In [ ]:
"""
Multilingual embedding-based topic assignment for wildlife posts.

Each post is embedded with a Sentence Transformers model (default: BAAI/bge-m3)
and assigned to the most similar of 12 standardized bilingual (English/Arabic)
topic descriptions, provided the cosine similarity clears a threshold.
"""

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm


# ============================================================
# CONFIGURATION  (edit these)
# ============================================================

INPUT_FILE = "wildlife_posts.csv"            # <-- path to your input data
OUTPUT_FILE = "Wildlife_12_topics_e5_large22.csv"
MESSAGE_COLUMN = "Message"
#MODEL_NAME = "BAAI/bge-m3"
MODEL_NAME = "intfloat/multilingual-e5-large-instruct"
SIMILARITY_THRESHOLD = 0.3                   # must be calibrated on human labels
BATCH_SIZE = 32
EXPECTED_NUM_TOPICS = 12
UNASSIGNED_LABEL = "UNASSIGNED"


# ============================================================
# 12 STANDARDIZED WILDLIFE TOPICS (English + Arabic)
# ============================================================

TOPIC_DEFINITIONS = {
    "Captivity, Zoos & Ethical Wildlife Management": """
Wild animals kept in zoos, aquariums, breeding facilities,
or captivity, including animal welfare, living conditions,
captive breeding, management, and ethical treatment.

الحيوانات البرية الموجودة في حدائق الحيوان أو الأحواض المائية
أو مراكز التكاثر أو الأسر، بما في ذلك رفاهية الحيوانات
وظروف الاحتجاز والتكاثر في الأسر والإدارة والمعاملة الأخلاقية.
""",

    "Climate Change Impact on Wildlife": """
Effects of climate change, global warming, extreme heat,
climate-related drought, changing weather patterns, and
other climatic changes on wild animals, their habitats,
migration, reproduction, and survival.

تأثير تغير المناخ والاحتباس الحراري وارتفاع درجات الحرارة
والجفاف المرتبط بالمناخ وتغير أنماط الطقس على الحيوانات
البرية وموائلها وهجرتها وتكاثرها وبقائها.
""",

    "Habitat Loss, Deforestation & Ecosystem Degradation": """
Destruction, fragmentation, or degradation of natural
habitats and ecosystems affecting wildlife, including
deforestation, land conversion, urban expansion,
terrestrial pollution, and loss of biodiversity.

تدمير الموائل الطبيعية وتجزئتها وتدهور النظم البيئية
التي تعتمد عليها الحياة البرية، بما في ذلك إزالة الغابات
وتغيير استخدام الأراضي والتوسع العمراني والتلوث البري
وفقدان التنوع البيولوجي.
""",

    "Wildlife Rehabilitation & Rescue": """
Rescue, veterinary treatment, rehabilitation, recovery,
and release of individual wild animals that are injured,
stranded, orphaned, trapped, or in distress.

إنقاذ الحيوانات البرية المصابة أو العالقة أو اليتيمة
أو المعرضة للخطر، وعلاجها بيطرياً وإعادة تأهيلها
وتعافيها وإطلاقها مجدداً في البرية.
""",

    "Wildlife Awareness, Media & Activism": """
Public education, awareness campaigns, environmental
advocacy, wildlife documentaries, photography, observation,
species information, animal behavior, and general
appreciation or promotion of wildlife and nature.

التوعية العامة والتثقيف البيئي وحملات حماية الحياة البرية
والنشاط البيئي والأفلام الوثائقية وتصوير الحيوانات
ومشاهدتها والمعلومات عن الأنواع وسلوك الحيوانات
والتقدير العام للحياة البرية والطبيعة.
""",

    "Human-Wildlife Conflict": """
Harmful or disputed interactions between wild animals
and humans, including attacks on people, crop damage,
livestock predation, retaliatory killing, property
damage, and efforts to improve human-wildlife coexistence.

الصراعات والتفاعلات الضارة بين الإنسان والحيوانات البرية،
بما في ذلك الهجمات على البشر وإتلاف المحاصيل وافتراس
الماشية والقتل الانتقامي والأضرار بالممتلكات والجهود
المبذولة لتحسين التعايش بين الإنسان والحياة البرية.
""",

    "Marine & Ocean Conservation": """
Protection and management of marine wildlife and aquatic
ecosystems, including whales, dolphins, sea turtles,
coral reefs, marine pollution, plastic waste, overfishing,
bycatch, and threats to marine biodiversity.

حماية وإدارة الحياة البرية البحرية والنظم البيئية المائية،
بما في ذلك الحيتان والدلافين والسلاحف البحرية والشعاب
المرجانية والتلوث البحري والنفايات البلاستيكية والصيد
الجائر والصيد العرضي والتهديدات للتنوع البيولوجي البحري.
""",

    "Endangered Species & Extinction Risk": """
Conservation status and survival of wildlife species,
including endangered and threatened species, declining
populations, extinction risk, species-level recovery,
and protection or reintroduction of threatened species.

حالة حفظ الأنواع البرية وبقائها، بما في ذلك الأنواع
المهددة بالانقراض وتناقص أعدادها وخطر انقراضها
وبرامج استعادة الأنواع وحمايتها وإعادة توطينها.
""",

    "Wildlife Crime, Poaching & Enforcement": """
Illegal hunting, killing, capturing, trading, or
trafficking of wild animals and wildlife products,
including poaching, illegal wildlife trade, investigations,
seizures, arrests, and enforcement against wildlife crime.

الصيد غير القانوني وقتل الحيوانات البرية وأسرها
والاتجار غير المشروع بها أو بمنتجاتها، بما في ذلك
الصيد الجائر وتهريب الحيوانات والتحقيقات والمصادرات
والاعتقالات وإنفاذ قوانين مكافحة جرائم الحياة البرية.
""",

    "Conservation Policy, Law & Funding": """
Government policies, legislation, regulations, funding,
donations, institutional programs, and activities by
conservation organizations intended to protect wildlife,
natural resources, and biodiversity.

السياسات الحكومية والتشريعات واللوائح والتمويل
والتبرعات والبرامج المؤسسية وأنشطة منظمات الحفاظ
على البيئة الهادفة إلى حماية الحياة البرية والموارد
الطبيعية والتنوع البيولوجي.
""",

    "Indigenous & Community-Based Conservation": """
Wildlife conservation involving indigenous peoples
or local communities, including traditional ecological
knowledge, community-led protection, local natural
resource management, and land rights connected to
biodiversity conservation.

حماية الحياة البرية بمشاركة السكان الأصليين أو المجتمعات
المحلية، بما في ذلك المعرفة البيئية التقليدية ومبادرات
الحماية المجتمعية وإدارة الموارد الطبيعية محلياً
وحقوق الأراضي المرتبطة بالحفاظ على التنوع البيولوجي.
""",

    "National Parks & Protected Areas": """
Establishment, management, conservation, and visitation
of national parks, nature reserves, wildlife sanctuaries,
and designated protected areas, including their ecological
importance and role in protecting wildlife.

إنشاء وإدارة وحماية وزيارة الحدائق الوطنية والمحميات
الطبيعية ومحميات الحياة البرية والمناطق المحمية،
بما في ذلك أهميتها البيئية ودورها في حماية الحيوانات البرية.
""",
}


# ============================================================
# TOPIC ASSIGNMENT
# ============================================================

def apply_sbert_topic_modeling(
    df,
    topic_definitions,
    message_column="Message",
    similarity_threshold=0.3,
    model_name="intfloat/multilingual-e5-large-instruct",
    #model_name="BAAI/bge-m3",
    batch_size=32,
    expected_num_topics=12,
    unassigned_label="UNASSIGNED",
):
    """
    Assign wildlife topics using multilingual embeddings.

    Parameters
    ----------
    df : pandas.DataFrame
        Input dataset containing social media messages.

    topic_definitions : dict
        Mapping of topic name -> bilingual topic description.

    message_column : str
        Column containing message text.

    similarity_threshold : float
        Minimum cosine similarity required for assignment.
        This threshold must be calibrated using human labels.

    model_name : str
        Sentence Transformers compatible embedding model.

    batch_size : int
        Number of posts encoded and classified per batch.

    expected_num_topics : int or None
        If set, raise an error unless exactly this many topics
        are supplied. Use None to skip the check.

    unassigned_label : str
        Label given to posts below the threshold or with no text.

    Returns
    -------
    pandas.DataFrame
        Copy of the input dataframe with additional topic columns:
        Assigned_Topic, SBERT_Best_Topic, Topic_Similarity,
        Second_Topic_Similarity, Topic_Similarity_Margin,
        Topic_Confidence.
    """

    print("=" * 90)
    print("MULTILINGUAL WILDLIFE TOPIC ASSIGNMENT".center(90))
    print("=" * 90)

    # --------------------------------------------------------
    # Step 1: Validate inputs
    # --------------------------------------------------------

    if message_column not in df.columns:
        raise ValueError(
            f"Column '{message_column}' not found. "
            f"Available columns: {list(df.columns)}"
        )

    if len(topic_definitions) < 2:
        raise ValueError(
            "At least 2 topic definitions are required."
        )

    if (
        expected_num_topics is not None
        and len(topic_definitions) != expected_num_topics
    ):
        raise ValueError(
            f"Expected exactly {expected_num_topics} topic "
            f"definitions, got {len(topic_definitions)}."
        )

    if not -1.0 <= similarity_threshold <= 1.0:
        raise ValueError(
            "Cosine similarity threshold must be "
            "between -1 and 1."
        )

    if batch_size < 1:
        raise ValueError("batch_size must be >= 1.")

    df = df.copy()

    topic_names = list(topic_definitions.keys())

    # Collapse line breaks / extra whitespace so the layout of
    # the definitions in the source file does not affect embeddings.
    topic_descriptions = [
        " ".join(str(description).split())
        for description in topic_definitions.values()
    ]

    empty_topics = [
        name
        for name, description in zip(topic_names, topic_descriptions)
        if not description
    ]
    if empty_topics:
        raise ValueError(
            f"Empty topic description(s): {empty_topics}"
        )

    print(f"\nTotal posts: {len(df):,}")
    print(f"Number of topics: {len(topic_names)}")
    print(f"Model: {model_name}")
    print(f"Similarity threshold: {similarity_threshold}")

    # --------------------------------------------------------
    # Step 2: Prepare messages
    # --------------------------------------------------------

    messages = (
        df[message_column]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    valid_positions = np.flatnonzero(
        messages.ne("").to_numpy()
    )

    print(f"Valid messages: {len(valid_positions):,}")
    print(
        f"Empty messages: "
        f"{len(df) - len(valid_positions):,}"
    )

    # --------------------------------------------------------
    # Step 3: Initialize output arrays
    # --------------------------------------------------------

    n = len(df)

    assigned_topics = np.full(n, unassigned_label, dtype=object)
    best_topics = np.full(n, None, dtype=object)
    topic_similarities = np.full(n, np.nan, dtype=np.float32)
    second_similarities = np.full(n, np.nan, dtype=np.float32)
    similarity_margins = np.full(n, np.nan, dtype=np.float32)

    # --------------------------------------------------------
    # Step 4: Load embedding model
    # --------------------------------------------------------

    print("\nLoading embedding model...")
    model = SentenceTransformer(model_name)
    print("Model loaded successfully.")

    # --------------------------------------------------------
    # Step 5: Encode standardized topic descriptions
    # --------------------------------------------------------

    print("\nEncoding topic descriptions...")

    topic_embeddings = model.encode(
        topic_descriptions,
        batch_size=batch_size,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False,
    ).astype(np.float32)

    topic_names_array = np.array(topic_names, dtype=object)

    print("Topic embeddings generated.")

    # --------------------------------------------------------
    # Step 6: Encode and classify posts in batches
    # --------------------------------------------------------

    print("\nEncoding and classifying messages...")

    batch_starts = range(0, len(valid_positions), batch_size)

    for start in tqdm(batch_starts, desc="Batches", unit="batch"):

        positions = valid_positions[start:start + batch_size]
        batch_messages = messages.iloc[positions].tolist()

        post_embeddings = model.encode(
            batch_messages,
            batch_size=batch_size,
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=False,
        ).astype(np.float32)

        # Cosine similarity: both sides are L2-normalized,
        # so the dot product is the cosine similarity.
        similarities = post_embeddings @ topic_embeddings.T

        # Highest and second-highest topic similarity per post
        sorted_indices = np.argsort(similarities, axis=1)
        best_indices = sorted_indices[:, -1]
        second_indices = sorted_indices[:, -2]

        rows = np.arange(len(positions))
        best_scores = similarities[rows, best_indices]
        second_scores = similarities[rows, second_indices]

        batch_topics = topic_names_array[best_indices]

        # Threshold-based classification
        is_assigned = best_scores >= similarity_threshold

        assigned_topics[positions] = np.where(
            is_assigned, batch_topics, unassigned_label
        )
        best_topics[positions] = batch_topics
        topic_similarities[positions] = best_scores
        second_similarities[positions] = second_scores
        similarity_margins[positions] = best_scores - second_scores

    # --------------------------------------------------------
    # Step 7: Store results
    # --------------------------------------------------------

    # Final label (UNASSIGNED if below threshold or empty text)
    df["Assigned_Topic"] = assigned_topics

    # Best semantic match, even if below threshold
    df["SBERT_Best_Topic"] = best_topics

    # Highest and second-highest cosine similarity
    df["Topic_Similarity"] = topic_similarities
    df["Second_Topic_Similarity"] = second_similarities

    # Difference between highest and second-highest scores
    df["Topic_Similarity_Margin"] = similarity_margins

    # Retained for compatibility with the original column name.
    # This is a similarity-margin heuristic,
    # NOT a calibrated classification probability.
    df["Topic_Confidence"] = similarity_margins

    # --------------------------------------------------------
    # Step 8: Print classification summary
    # --------------------------------------------------------

    print("\n" + "=" * 90)
    print("TOPIC ASSIGNMENT SUMMARY".center(90))
    print("=" * 90)

    topic_counts = df["Assigned_Topic"].value_counts()

    print(f"\n{'Topic':<55} {'Count':>10} {'%':>8}")
    print("-" * 75)

    for topic, count in topic_counts.items():
        pct = 100 * count / n if n else 0
        print(f"{topic:<55} {count:>10,} {pct:>7.2f}%")

    assigned_count = int(
        (df["Assigned_Topic"] != unassigned_label).sum()
    )
    coverage = 100 * assigned_count / n if n else 0

    print(
        f"\nAssignment coverage: {coverage:.2f}% "
        f"({assigned_count:,} of {n:,} posts)"
    )

    valid_scores = topic_similarities[valid_positions]
    if len(valid_scores):
        print(
            "Best-topic similarity (non-empty posts): "
            f"min={valid_scores.min():.3f}, "
            f"median={np.median(valid_scores):.3f}, "
            f"max={valid_scores.max():.3f}"
        )

    print("=" * 90)

    return df


# ============================================================
# RUN BGE-M3 TOPIC ASSIGNMENT
# ============================================================

if __name__ == "__main__":

    #df = pd.read_csv(INPUT_FILE)

    df_with_topics = apply_sbert_topic_modeling(
        df=df, #load the dataset as dataframe and asssign
        topic_definitions=TOPIC_DEFINITIONS,
        message_column=MESSAGE_COLUMN,
        similarity_threshold=SIMILARITY_THRESHOLD,
        model_name=MODEL_NAME,
        batch_size=BATCH_SIZE,
        expected_num_topics=EXPECTED_NUM_TOPICS,
        unassigned_label=UNASSIGNED_LABEL,
    )

    df_with_topics.to_csv(
        OUTPUT_FILE,
        index=False,
        encoding="utf-8-sig",
    )

    print(
        f"Saved {len(df_with_topics):,} posts to {OUTPUT_FILE}"
    )

In [ ]:

import numpy as np
import pandas as pd

thresholds = np.arange(0.30, 0.61, 0.025)

results = []

for threshold in thresholds:

    assigned = (
        df_with_topics["Topic_Similarity"] >= threshold
    ).sum()

    unassigned = len(df_with_topics) - assigned

    results.append({
        "Threshold": round(float(threshold), 3),
        "Assigned": assigned,
        "Unassigned": unassigned,
        "Assignment_Rate": round(
            assigned / len(df_with_topics) * 100, 2
        ),
        "Unassigned_Rate": round(
            unassigned / len(df_with_topics) * 100, 2
        )
    })

threshold_results = pd.DataFrame(results)

print(threshold_results.to_string(index=False))

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score
import pandas as pd

# human_labels:
# Human-verified topic labels, including UNASSIGNED

y_true = (
    df_e5["Human_Topic"] != "UNASSIGNED"
).astype(int)

models = {
    "E5-large": df_e5["Topic_Similarity"],
    "BGE-M3": df_bge["Topic_Similarity"]
}

results = []

for name, scores in models.items():

    valid = scores.notna() & y_true.notna()

    if y_true[valid].nunique() < 2:
        raise ValueError(
            "Both relevant and UNASSIGNED human labels "
            "are required to calculate AUROC."s
        )

    results.append({
        "Model": name,
        "AUROC": roc_auc_score(
            y_true[valid],
            scores[valid]
        ),
        "Average_Precision": average_precision_score(
            y_true[valid],
            scores[valid]
        )
    })

print(pd.DataFrame(results))

In [ ]:

import matplotlib.pyplot as plt

fig, axes = plt.subplots(
    1, 2,
    figsize=(7.0, 2.8)
)

model_scores = {
    "E5-Large": "E5_Similarity",
    "BGE-M3": "BGE_Similarity"
}

for name, column in model_scores.items():

    valid = comparison[column].notna()

    y = comparison.loc[valid, "Relevant"]
    scores = comparison.loc[valid, column]

    # ROC curve
    fpr, tpr, _ = roc_curve(y, scores)

    auc = roc_auc_score(y, scores)

    axes[0].plot(
        fpr,
        tpr,
        linewidth=1.7,
        label=f"{name} (AUC={auc:.3f})"
    )

    # Precision-recall curve
    precision, recall, _ = precision_recall_curve(
        y, scores
    )

    ap = average_precision_score(y, scores)

    axes[1].plot(
        recall,
        precision,
        linewidth=1.7,
        label=f"{name} (AP={ap:.3f})"
    )


# Random classifier reference
axes[0].plot(
    [0, 1], [0, 1],
    linestyle="--",
    color="gray",
    linewidth=1
)

# AP prevalence baseline
baseline = comparison["Relevant"].mean()

axes[1].axhline(
    baseline,
    linestyle="--",
    color="gray",
    linewidth=1,
    label=f"Prevalence={baseline:.3f}"
)

axes[0].set_xlabel("False positive rate")
axes[0].set_ylabel("True positive rate")
axes[0].set_title("(a) ROC")

axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("(b) Precision–Recall")

for ax in axes:
    ax.legend(fontsize=7, frameon=False)
    ax.grid(axis="both", alpha=0.2)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.02)

plt.tight_layout()

plt.savefig(
    "embedding_relevance_roc_pr.pdf",
    bbox_inches="tight"
)

plt.savefig(
    "embedding_relevance_roc_pr.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()